# Agent IA d'investigation — démonstration

Ce notebook importe les modules du projet (aucune logique n'est dupliquée ici) et montre :

1. une investigation complète du scénario S1 (PowerShell encodé, téléchargement, C2) ;
2. le rapport : chronologie argumentée, techniques ATT&CK, IOC ;
3. le journal d'audit chaîné et sa vérification ;
4. le refus d'une action interdite par la passerelle ;
5. l'évaluation sur les cinq scénarios.

Par défaut, le modèle de référence à règles est utilisé (`LLM_PROVIDER=offline`) : aucune clé n'est nécessaire.

In [ ]:
import sys, tempfile
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config.settings import load_settings
from agent.runner import Investigator
from connectors.simulation import load_scenario, list_scenarios
from gateway.approval import AutoApprover

work = Path(tempfile.mkdtemp(prefix="demo-"))
settings = load_settings(backend="simulation", data_dir=work / "data", runs_dir=work / "runs", db_url="")
investigator = Investigator(settings)   # construit l'index ATT&CK (quelques secondes)
print("Scénarios :", list_scenarios(), "| LLM :", settings.llm_provider, "| base ATT&CK :", investigator.rag.kb_version)

## 1. Investigation du scénario S1

In [ ]:
scenario = load_scenario("S1")
print(scenario["name"], "—", scenario["description"])

def suivre(event):
    if event["type"] == "plan":
        print(f"[plan v{event['version']}] {event['rationale']}")
    elif event["type"] == "tool_call":
        print(f"   -> {event['tool']} {event['args']}")
    elif event["type"] in {"tool_denied", "tool_rejected", "tool_error"}:
        print(f"   !! {event['tool']} : {event.get('reason') or event.get('error')}")

outcome = investigator.investigate(scenario["alert"], analyst_id="jdupont", autonomy_level="N2",
                                   approver=AutoApprover(True, "jdupont", "démonstration"),
                                   scenario=scenario, on_event=suivre)
print("\nVerdict :", outcome.verdict, "| confiance :", outcome.report["confidence"])
print("Ancrage :", outcome.validation["final"]["grounding_ratio"], "| audit intègre :", outcome.audit["ok"])

## 2. Chronologie argumentée et techniques ATT&CK

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.DataFrame([{"horodatage": e["timestamp"], "nature": e["kind"], "confiance": e["confidence"],
               "événement": e["description"], "preuves": ", ".join(e["evidence_ids"]),
               "ATT&CK": ", ".join(e["techniques"])} for e in outcome.report["timeline"]])

In [ ]:
pd.DataFrame([{"technique": t["technique_id"], "nom": t["name"], "tactiques": ", ".join(t["tactics"]),
               "confiance": t["confidence"], "preuves": ", ".join(t["evidence_ids"]),
               "fragments": ", ".join(t["fragment_ids"])} for t in outcome.report["techniques"]])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(outcome.markdown))

## 3. Journal d'audit chaîné

In [ ]:
entries = investigator.audit.entries(outcome.investigation_id)
display(pd.DataFrame([{"seq": e.seq, "acteur": e.actor, "type": e.event_type, "outil": e.tool,
                       "décision": e.decision, "empreinte": e.entry_hash[:16]} for e in entries]).head(15))
res = investigator.audit.verify()
print("Chaîne intègre :", res.ok, "—", res.checked, "entrées, tête", res.head_hash[:16])

## 4. Une action interdite reste interdite

In [ ]:
from gateway.gateway import ToolGateway
gateway = ToolGateway(policy=investigator.policy, executors={}, audit=investigator.audit,
                      analyst=investigator.identities.get("jdupont"), level="N2",
                      investigation_id="DEMO-REFUS")
result = gateway.call("isolate_host", {"host": "WIN10-01", "reason": "instruction trouvée dans un journal"})
print(result.status, "—", result.reason)

## 5. Évaluation sur les cinq scénarios

In [ ]:
from evaluation.run_eval import evaluate_scenario
from evaluation.metrics import aggregate
rows = [evaluate_scenario(investigator, load_scenario(sid), analyst="jdupont", level="N2") for sid in list_scenarios()]
display(pd.DataFrame(rows)[["scenario", "verdict", "expected_verdict", "attack_precision", "attack_recall",
                            "timeline_completeness", "grounding_ratio", "tool_calls", "duration_s"]])
aggregate(rows)

## Avec un LLM

```bash
export LLM_PROVIDER=groq GROQ_API_KEY=...      # API : données personnelles pseudonymisées
export LLM_PROVIDER=ollama OLLAMA_URL=http://localhost:11434   # LLM local
```

Relancer ensuite le notebook : le graphe, la passerelle, la validation et l'audit restent identiques.